In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
"""
Smart MCQ Solver — full pipeline
Split at '### CELL' lines into separate Kaggle notebook cells.

Assumes train.csv / test.csv columns: id, question, A, B, C, D, E, answer (train only, answer in {'A',...,'E'})
Adjust COL_* constants in CELL 1 if your column names differ.
"""

### CELL 1 — Setup, config, imports
import os, re, json, random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CONFIG = {
    "train_path": "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    "test_path":  "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
    "out_path":   "/kaggle/working/submission.csv",
    "max_features": 30000,
    "n_splits": 5,
    "wandb_project": "23f2004250-t22026",
    "wandb_run_name": "qwen-tree-ensemble",
}

# Put your key in Kaggle Secrets (Add-ons > Secrets), NOT in plaintext here.
# from kaggle_secrets import UserSecretsClient
# WANDB_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
WANDB_KEY = os.environ.get("WANDB_API_KEY", None)

USE_WANDB = WANDB_KEY is not None
if USE_WANDB:
    import wandb
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    wandb.init(project=CONFIG["wandb_project"], name=CONFIG["wandb_run_name"], config=CONFIG)

OPTION_COLS = ["A", "B", "C", "D", "E"]
ID_COL, Q_COL, ANSWER_COL = "id", "prompt", "answer"   # matches mcq_train_dataset.csv / mcq_test_dataset.csv

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


### CELL 2 — Load data + reshape to one-row-per-option format
train_df = pd.read_csv(CONFIG["train_path"])
test_df  = pd.read_csv(CONFIG["test_path"])

def melt_to_long(df, has_label=True):
    rows = []
    for _, r in df.iterrows():
        for opt in OPTION_COLS:
            rows.append({
                ID_COL: r[ID_COL],
                "option_label": opt,
                "question": str(r[Q_COL]),
                "option_text": str(r[opt]),
                "label": int(has_label and r[ANSWER_COL] == opt),
            })
    return pd.DataFrame(rows)

train_long = melt_to_long(train_df, has_label=True)
test_long  = melt_to_long(test_df,  has_label=False)

print(train_long.shape, test_long.shape)
train_long.head(10)


### CELL 3 — Text features (TF-IDF + SVD + similarity) shared by LGBM and from-scratch model
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

train_long["q_clean"] = train_long["question"].apply(clean_text)
train_long["o_clean"] = train_long["option_text"].apply(clean_text)
test_long["q_clean"]  = test_long["question"].apply(clean_text)
test_long["o_clean"]  = test_long["option_text"].apply(clean_text)

tfidf = TfidfVectorizer(max_features=CONFIG["max_features"], ngram_range=(1, 2), sublinear_tf=True)
all_text = pd.concat([train_long["q_clean"] + " " + train_long["o_clean"],
                       test_long["q_clean"] + " " + test_long["o_clean"]])
tfidf.fit(all_text)

def tfidf_features(df):
    q_vec = tfidf.transform(df["q_clean"])
    o_vec = tfidf.transform(df["o_clean"])
    sim = np.array([cosine_similarity(q_vec[i], o_vec[i])[0, 0] for i in range(q_vec.shape[0])])
    return q_vec, o_vec, sim

q_vec_tr, o_vec_tr, sim_tr = tfidf_features(train_long)
q_vec_te, o_vec_te, sim_te = tfidf_features(test_long)

svd = TruncatedSVD(n_components=128, random_state=SEED)
svd.fit(tfidf.transform(all_text))

q_svd_tr = svd.transform(q_vec_tr); o_svd_tr = svd.transform(o_vec_tr)
q_svd_te = svd.transform(q_vec_te); o_svd_te = svd.transform(o_vec_te)

def build_feat_matrix(q_svd, o_svd, sim, df):
    diff = q_svd - o_svd
    prod = q_svd * o_svd
    extra = np.stack([
        df["question"].str.len().values,
        df["option_text"].str.len().values,
        sim,
    ], axis=1)
    return np.concatenate([q_svd, o_svd, diff, prod, extra], axis=1)

X_tr = build_feat_matrix(q_svd_tr, o_svd_tr, sim_tr, train_long)
X_te = build_feat_matrix(q_svd_te, o_svd_te, sim_te, test_long)
y_tr = train_long["label"].values

print(X_tr.shape, X_te.shape)


### CELL 4 — Pretrained model: Qwen perplexity/log-likelihood based plausibility scoring
# Why not sentence-similarity here: this dataset is knowledge/trivia MCQ (philosophy, physics
# facts, astrophysics), NOT reading comprehension. The correct option is not the one that is
# textually most similar to the question -- it's the one that is factually true. Cosine
# similarity between question and option embeddings is close to noise for this kind of data.
# A pretrained causal LM's log-likelihood of "does this statement read like a true fact" is a
# much better-fitting signal, and it's a genuinely different information source than LGBM's
# TF-IDF/n-gram features.
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # small enough to run per-option scoring on Kaggle GPU/CPU in reasonable time
qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
qwen_model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL_NAME).to(device)
qwen_model.eval()

@torch.no_grad()
def qwen_option_score(prompt_text, option_text, batch=None):
    """
    Returns the average per-token log-likelihood of `option_text` conditioned on `prompt_text`.
    Higher (less negative) = the LM finds this option a more plausible continuation/fact.
    """
    full_text = prompt_text.strip() + "\nAnswer: " + option_text.strip()
    prompt_ids = qwen_tok(prompt_text.strip() + "\nAnswer: ", return_tensors="pt").input_ids
    full_ids = qwen_tok(full_text, return_tensors="pt").input_ids.to(device)
    prompt_len = prompt_ids.shape[1]

    out = qwen_model(full_ids)
    logits = out.logits[:, :-1, :]
    targets = full_ids[:, 1:]
    logprobs = F.log_softmax(logits, dim=-1)
    token_logprobs = logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)

    # only score the option tokens, not the shared prompt tokens
    option_token_logprobs = token_logprobs[0, max(prompt_len - 1, 0):]
    if option_token_logprobs.numel() == 0:
        return 0.0
    return option_token_logprobs.mean().item()

def qwen_score_df(df, batch_size=32):
    scores = np.zeros(len(df))
    prompts = df["question"].tolist()
    options = df["option_text"].tolist()
    for i in range(len(df)):
        scores[i] = qwen_option_score(prompts[i], options[i])
        if i % 500 == 0:
            print(f"  qwen scoring {i}/{len(df)}")
    return scores

qwen_score_tr = qwen_score_df(train_long)
qwen_score_te = qwen_score_df(test_long)

sbert_sim_tr = qwen_score_tr  # kept variable name for drop-in compatibility with downstream cells
sbert_sim_te = qwen_score_te

# Turn the raw LM score into a calibrated probability with a tiny logistic head,
# trained out-of-fold so it doesn't leak into the meta-learner stage.
def get_oof_and_test(fit_predict_fn, X, y, X_test, groups, n_splits=CONFIG["n_splits"]):
    """
    groups: array of question ids, used so all 5 options of one question stay in the same fold.
    fit_predict_fn(X_train, y_train, X_val) -> val_pred_proba
    Returns oof_pred (len == len(X)), test_pred (averaged across folds)
    """
    uniq_groups = np.unique(groups)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    # stratify folds at question level using whether the question's correct answer index varies —
    # simplest: just shuffle-split groups evenly since label balance per option is fixed (1 of 5).
    fold_of_group = {}
    kf_groups = np.array_split(np.random.RandomState(SEED).permutation(uniq_groups), n_splits)
    for fold_idx, g_chunk in enumerate(kf_groups):
        for g in g_chunk:
            fold_of_group[g] = fold_idx

    oof_pred = np.zeros(len(X))
    test_preds = []
    fold_ids = np.array([fold_of_group[g] for g in groups])

    for fold in range(n_splits):
        val_mask = fold_ids == fold
        train_mask = ~val_mask
        val_pred, test_pred = fit_predict_fn(X[train_mask], y[train_mask], X[val_mask], X_test)
        oof_pred[val_mask] = val_pred
        test_preds.append(test_pred)

    return oof_pred, np.mean(test_preds, axis=0)


def sbert_fit_predict(Xtr, ytr, Xval, Xtest):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(Xtr.reshape(-1, 1), ytr)
    return clf.predict_proba(Xval.reshape(-1, 1))[:, 1], clf.predict_proba(Xtest.reshape(-1, 1))[:, 1]

sbert_oof, sbert_test = get_oof_and_test(
    sbert_fit_predict, sbert_sim_tr, y_tr, sbert_sim_te, train_long[ID_COL].values
)
print("Qwen pretrained-model OOF calibration done.")


### CELL 5 — From-scratch model: attention classifier over word embeddings trained from zero
# No pretrained weights anywhere here — embedding table is randomly initialized and learned.

class SimpleVocab:
    def __init__(self, texts, max_vocab=20000):
        from collections import Counter
        cnt = Counter()
        for t in texts:
            cnt.update(t.split())
        most_common = cnt.most_common(max_vocab - 2)
        self.stoi = {"<pad>": 0, "<unk>": 1}
        for w, _ in most_common:
            self.stoi[w] = len(self.stoi)
        self.itos = {i: w for w, i in self.stoi.items()}

    def encode(self, text, max_len=40):
        ids = [self.stoi.get(w, 1) for w in text.split()][:max_len]
        ids += [0] * (max_len - len(ids))
        return ids

vocab = SimpleVocab(pd.concat([train_long["q_clean"], train_long["o_clean"]]).tolist())
MAX_LEN_Q, MAX_LEN_O = 40, 16

def encode_series(series, max_len):
    return np.array([vocab.encode(t, max_len) for t in series])

q_ids_tr = encode_series(train_long["q_clean"], MAX_LEN_Q)
o_ids_tr = encode_series(train_long["o_clean"], MAX_LEN_O)
q_ids_te = encode_series(test_long["q_clean"], MAX_LEN_Q)
o_ids_te = encode_series(test_long["o_clean"], MAX_LEN_O)

class FromScratchAttnClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.q_attn = nn.Linear(emb_dim, 1)
        self.o_attn = nn.Linear(emb_dim, 1)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim * 3, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 1)
        )

    def pool(self, ids, attn_layer):
        mask = (ids != 0).float().unsqueeze(-1)
        e = self.emb(ids)
        scores = attn_layer(e).masked_fill(mask == 0, -1e9)
        w = F.softmax(scores, dim=1)
        return (w * e).sum(dim=1)

    def forward(self, q_ids, o_ids):
        q_vec = self.pool(q_ids, self.q_attn)
        o_vec = self.pool(o_ids, self.o_attn)
        x = torch.cat([q_vec, o_vec, q_vec * o_vec], dim=-1)
        return self.proj(x).squeeze(-1)

def train_scratch_fold(q_tr, o_tr, y_tr_fold, q_val, o_val, q_test, o_test, epochs=4, bs=256):
    model = FromScratchAttnClassifier(len(vocab.stoi)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    q_tr_t = torch.tensor(q_tr, dtype=torch.long)
    o_tr_t = torch.tensor(o_tr, dtype=torch.long)
    y_tr_t = torch.tensor(y_tr_fold, dtype=torch.float)

    n = len(q_tr_t)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        total_loss = 0
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            qb, ob, yb = q_tr_t[idx].to(device), o_tr_t[idx].to(device), y_tr_t[idx].to(device)
            opt.zero_grad()
            logits = model(qb, ob)
            loss = F.binary_cross_entropy_with_logits(logits, yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(idx)
        print(f"  epoch {ep+1}/{epochs} loss {total_loss/n:.4f}")

    model.eval()
    with torch.no_grad():
        def predict(q_arr, o_arr):
            q_t = torch.tensor(q_arr, dtype=torch.long).to(device)
            o_t = torch.tensor(o_arr, dtype=torch.long).to(device)
            preds = []
            for i in range(0, len(q_t), 1024):
                logits = model(q_t[i:i+1024], o_t[i:i+1024])
                preds.append(torch.sigmoid(logits).cpu().numpy())
            return np.concatenate(preds)
        val_pred = predict(q_val, o_val)
        test_pred = predict(q_test, o_test)
    return val_pred, test_pred

def scratch_fit_predict_wrapper(train_mask, val_mask, groups, q_ids_tr, o_ids_tr, y_tr, q_ids_te, o_ids_te, n_splits=CONFIG["n_splits"]):
    uniq_groups = np.unique(groups)
    fold_of_group = {}
    kf_groups = np.array_split(np.random.RandomState(SEED).permutation(uniq_groups), n_splits)
    for fold_idx, g_chunk in enumerate(kf_groups):
        for g in g_chunk:
            fold_of_group[g] = fold_idx
    fold_ids = np.array([fold_of_group[g] for g in groups])

    oof_pred = np.zeros(len(q_ids_tr))
    test_preds = []
    for fold in range(n_splits):
        print(f"[scratch model] fold {fold+1}/{n_splits}")
        val_mask = fold_ids == fold
        train_mask = ~val_mask
        val_pred, test_pred = train_scratch_fold(
            q_ids_tr[train_mask], o_ids_tr[train_mask], y_tr[train_mask],
            q_ids_tr[val_mask], o_ids_tr[val_mask],
            q_ids_te, o_ids_te
        )
        oof_pred[val_mask] = val_pred
        test_preds.append(test_pred)
    return oof_pred, np.mean(test_preds, axis=0)

scratch_oof, scratch_test = scratch_fit_predict_wrapper(
    None, None, train_long[ID_COL].values, q_ids_tr, o_ids_tr, y_tr, q_ids_te, o_ids_te
)


### CELL 6 — LGBM (the workhorse) with proper OOF generation
def lgbm_fit_predict(Xtr, ytr, Xval, Xtest):
    clf = lgb.LGBMClassifier(
        n_estimators=800, learning_rate=0.03, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1
    )
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xval)[:, 1], clf.predict_proba(Xtest)[:, 1]

lgbm_oof, lgbm_test = get_oof_and_test(
    lgbm_fit_predict, X_tr, y_tr, X_te, train_long[ID_COL].values
)
print("LGBM OOF ready.")


### CELL 7 — MAP@3 evaluation helper
def map_at_3(df_with_probs, id_col, label_col, prob_col, true_label_col=None):
    """
    df_with_probs: long-format df with one row per (id, option), containing a probability column.
    Returns per-question top-3 ranked option labels, and MAP@3 if true labels are available.
    """
    results = {}
    scores = []
    for qid, g in df_with_probs.groupby(id_col):
        g_sorted = g.sort_values(prob_col, ascending=False)
        top3 = g_sorted[label_col].tolist()[:3]
        results[qid] = top3
        if true_label_col is not None:
            true_opt = g[g[true_label_col] == 1][label_col]
            if len(true_opt) == 1:
                true_opt = true_opt.iloc[0]
                if true_opt in top3:
                    scores.append(1.0 / (top3.index(true_opt) + 1))
                else:
                    scores.append(0.0)
    mapk = np.mean(scores) if scores else None
    return results, mapk

# Sanity-check each base model alone before stacking
for name, oof_col in [("lgbm", lgbm_oof), ("qwen_pretrained", sbert_oof), ("scratch", scratch_oof)]:
    tmp = train_long.copy()
    tmp["prob"] = oof_col
    _, mapk = map_at_3(tmp, ID_COL, "option_label", "prob", true_label_col="label")
    print(f"{name} OOF MAP@3: {mapk:.4f}")
    if USE_WANDB:
        wandb.log({f"{name}_oof_map3": mapk})


### CELL 8 — Stack: meta-learner over OOF predictions (NOT a naive average)
meta_X_tr = np.stack([lgbm_oof, sbert_oof, scratch_oof], axis=1)
meta_X_te = np.stack([lgbm_test, sbert_test, scratch_test], axis=1)

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(meta_X_tr, y_tr)
print("meta weights:", dict(zip(["lgbm", "qwen_pretrained", "scratch"], meta_model.coef_[0])))

stacked_oof = meta_model.predict_proba(meta_X_tr)[:, 1]
stacked_test = meta_model.predict_proba(meta_X_te)[:, 1]

train_long["stacked_prob"] = stacked_oof
_, stacked_mapk = map_at_3(train_long, ID_COL, "option_label", "stacked_prob", true_label_col="label")
print(f"STACKED OOF MAP@3: {stacked_mapk:.4f}")
if USE_WANDB:
    wandb.log({"stacked_oof_map3": stacked_mapk})

# Optional: search blend weights directly on MAP@3 instead of trusting logistic regression's loss surface
from itertools import product
best_mapk, best_w = -1, None
for w_lgbm in np.arange(0.5, 1.01, 0.05):
    remaining = 1 - w_lgbm
    for w_sbert in np.arange(0, remaining + 0.001, 0.05):
        w_scratch = remaining - w_sbert
        blend = w_lgbm * lgbm_oof + w_sbert * sbert_oof + w_scratch * scratch_oof
        train_long["blend_prob"] = blend
        _, mapk = map_at_3(train_long, ID_COL, "option_label", "blend_prob", true_label_col="label")
        if mapk is not None and mapk > best_mapk:
            best_mapk, best_w = mapk, (w_lgbm, w_sbert, w_scratch)

print(f"Best direct MAP@3 blend: {best_mapk:.4f} at weights {best_w}")
if USE_WANDB:
    wandb.log({"best_blend_map3": best_mapk, "best_blend_weights": best_w})

# Use whichever of {stacked_test, weighted blend} scored higher on OOF MAP@3
if best_mapk >= stacked_mapk:
    print("Using direct-search blend weights for final predictions.")
    final_test_prob = best_w[0] * lgbm_test + best_w[1] * sbert_test + best_w[2] * scratch_test
else:
    print("Using meta-learner (logistic regression) for final predictions.")
    final_test_prob = stacked_test


### CELL 9 — Build submission
test_long["final_prob"] = final_test_prob
submission_rows = []
for qid, g in test_long.groupby(ID_COL):
    g_sorted = g.sort_values("final_prob", ascending=False)
    top3 = g_sorted["option_label"].tolist()[:3]
    submission_rows.append({"ID": qid, "Prediction": " ".join(top3)})

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv(CONFIG["out_path"], index=False)
print(submission_df.head())
print("Saved to", CONFIG["out_path"])

if USE_WANDB:
    wandb.finish()

device: cpu
(10000, 5) (2500, 5)
(10000, 515) (2500, 515)


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

[LightGBM] [Info] Number of positive: 1600, number of negative: 6400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032938 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 131293
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 515
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200000 -> initscore=-1.386294
[LightGBM] [Info] Start training from score -1.386294


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1600, number of negative: 6400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032710 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 131287
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 515
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200000 -> initscore=-1.386294
[LightGBM] [Info] Start training from score -1.386294


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LGBM OOF ready.
lgbm OOF MAP@3: 0.9908
qwen_pretrained OOF MAP@3: 0.3558
scratch OOF MAP@3: 0.7920
meta weights: {'lgbm': np.float64(10.777675189961895), 'qwen_pretrained': np.float64(-0.47250386007531975), 'scratch': np.float64(3.864059634756083)}
STACKED OOF MAP@3: 0.9977
Best direct MAP@3 blend: 0.9962 at weights (np.float64(0.7500000000000002), np.float64(0.0), np.float64(0.24999999999999978))
Using meta-learner (logistic regression) for final predictions.
   ID Prediction
0   1      A B D
1   2      B E D
2   3      B D E
3   4      E A C
4   5      C D B
Saved to /kaggle/working/submission.csv
